# 🐼 Pandas Practice — Expert Exercises (21–40)

Advanced pandas patterns used in real data engineering and analysis work.

| Exercise | Topic |
|----------|-------|
| 21 | Time Series Resampling |
| 22 | crosstab() & Contingency Tables |
| 23 | Custom Aggregation with agg() |
| 24 | where() & mask() — Conditional Updates |
| 25 | merge() with indicator & many-to-many |
| 26 | wide_to_long() |
| 27 | Advanced String Ops (regex extract, findall) |
| 28 | DateTime Advanced (tz, offsets, holidays) |
| 29 | Chunked CSV Processing |
| 30 | Rolling & Expanding Advanced |
| 31 | groupby + transform (broadcast) |
| 32 | apply() axis=1 vs vectorized |
| 33 | Hierarchical GroupBy |
| 34 | merge_asof() — Time-Based Join |
| 35 | pivot_table() with margins & aggfunc |
| 36 | DataFrame.compare() |
| 37 | pd.json_normalize() — Nested JSON |
| 38 | IntervalIndex & pd.cut advanced |
| 39 | GroupBy + cumulative stats |
| 40 | Mini Project: End-to-End Sales Pipeline |

---

## Setup

In [ ]:
import pandas as pd
import numpy as np
np.random.seed(42)
print('pandas:', pd.__version__)

---
## Exercise 21 — Time Series Resampling

**Scenario:** You have hourly sensor readings for 60 days. Analyse at multiple time scales.

Tasks:
1. Resample to **daily** — mean temperature, total rainfall
2. Resample to **weekly** — max temperature
3. Resample to **monthly** — mean and std of temperature
4. Use `asfreq('D')` to get daily frequency (no aggregation) — note the NaNs
5. Forward-fill missing values after `asfreq()` and compare with `interpolate()`

In [ ]:
idx = pd.date_range('2024-01-01', periods=60*24, freq='h')
np.random.seed(7)
sensors = pd.DataFrame({
    'temperature': 20 + 10*np.sin(np.linspace(0, 60*2*np.pi/24, len(idx))) + np.random.normal(0, 2, len(idx)),
    'rainfall':    np.random.exponential(0.5, len(idx)).round(2),
}, index=idx)
print(sensors.head(3))

# 1. Daily mean temp, total rainfall
# YOUR CODE HERE

# 2. Weekly max temperature
# YOUR CODE HERE

# 3. Monthly mean + std of temperature
# YOUR CODE HERE

# 4. asfreq('D') on daily data (use daily resampled from task 1)
# YOUR CODE HERE

# 5. ffill vs interpolate on a sparse daily series
sparse = sensors['temperature'].resample('D').mean()
sparse_missing = sparse.copy()
sparse_missing.iloc[3:6] = np.nan   # introduce gaps
# YOUR CODE HERE

---
## Exercise 22 — crosstab() & Contingency Tables

**Scenario:** HR analysis — explore promotion rates across department and gender.

Tasks:
1. Build a crosstab of `dept` vs `promoted` (counts)
2. Normalise by row to get promotion **rate** per department
3. Add row and column **margins** (totals)
4. Build a crosstab of `dept` vs `gender` with `promoted` as the **values** and `mean` as aggfunc
5. Find the department with the highest female promotion rate

In [ ]:
np.random.seed(21)
n = 1000
hr = pd.DataFrame({
    'employee_id': range(n),
    'dept':     np.random.choice(['Eng','Sales','HR','Finance','Ops'], n),
    'gender':   np.random.choice(['M','F'], n, p=[0.55, 0.45]),
    'level':    np.random.choice(['Junior','Mid','Senior'], n, p=[0.4,0.4,0.2]),
    'promoted': np.random.choice([0, 1], n, p=[0.75, 0.25]),
})

# 1. Basic crosstab
# YOUR CODE HERE

# 2. Row-normalised (promotion rate per dept)
# YOUR CODE HERE

# 3. Add margins
# YOUR CODE HERE

# 4. crosstab dept x gender, values=promoted, aggfunc=mean
# YOUR CODE HERE

# 5. Dept with highest female promotion rate
# YOUR CODE HERE

---
## Exercise 23 — Custom Aggregation with agg()

**Scenario:** Compute rich summary statistics for a sales dataset in one pass.

Tasks:
1. Group by `region` and use a **dict agg** to compute: `revenue` → sum, mean, max; `quantity` → sum, mean
2. Use **named aggregations** (pandas ≥ 0.25) to get flat column names
3. Write a custom function `iqr(s)` returning the interquartile range and apply it via `agg`
4. Apply multiple custom + built-in functions to `revenue` using a list in `agg`
5. Use `agg` with a lambda to compute coefficient of variation (`std/mean*100`) per region

In [ ]:
np.random.seed(11)
n = 2000
sales = pd.DataFrame({
    'region':   np.random.choice(['North','South','East','West'], n),
    'product':  np.random.choice(['A','B','C','D'], n),
    'revenue':  np.random.exponential(500, n).round(2),
    'quantity': np.random.randint(1, 50, n),
    'discount': np.random.uniform(0, 0.4, n).round(3),
})

# 1. Dict agg
# YOUR CODE HERE

# 2. Named aggregation (no MultiIndex columns)
# YOUR CODE HERE

# 3. Custom IQR function
def iqr(s):
    return s.quantile(0.75) - s.quantile(0.25)
# YOUR CODE HERE

# 4. List of functions applied to revenue
# YOUR CODE HERE

# 5. Coefficient of variation
# YOUR CODE HERE

---
## Exercise 24 — where() & mask() — Conditional Updates

**Scenario:** Apply business rules to clean and cap sensor and financial data.

Tasks:
1. Use `where()` to keep values only where `temperature` is between -10 and 50, else set to `NaN`
2. Use `mask()` to replace `revenue` values below 0 with 0
3. Use `where()` to cap `revenue` at the 99th percentile
4. Use `np.where()` to create a `price_tier` column: `'high'` if price > 100, else `'low'`
5. Chain two `mask()` calls to clean both outlier ends of a distribution

In [ ]:
np.random.seed(24)
n = 500
df = pd.DataFrame({
    'temperature': np.random.normal(20, 30, n).round(1),   # some outliers
    'revenue':     np.random.normal(500, 200, n).round(2), # some negatives
    'price':       np.random.uniform(10, 200, n).round(2),
})
print('Revenue negatives before:', (df['revenue'] < 0).sum())
print('Temp outliers before:', ((df['temperature'] < -10) | (df['temperature'] > 50)).sum())

# 1. where() on temperature
# YOUR CODE HERE

# 2. mask() on revenue < 0 → 0
# YOUR CODE HERE

# 3. Cap revenue at 99th percentile
# YOUR CODE HERE

# 4. price_tier using np.where
# YOUR CODE HERE

# 5. Chain mask: remove bottom 1% and top 1% of revenue
# YOUR CODE HERE

---
## Exercise 25 — merge() with indicator & Many-to-Many

**Scenario:** Analyse customer orders and tag associations.

Tasks:
1. Left-join `orders` with `customers` on `customer_id`, add `indicator=True`
2. Use the `_merge` column to find orders with no matching customer record
3. Perform a many-to-many join: merge `products` with `product_tags` (each product can have many tags)
4. After the many-to-many join, count products per tag
5. Use `validate='many_to_one'` on a join that should be many-to-one — catch the error if it isn't

In [ ]:
np.random.seed(25)
customers = pd.DataFrame({
    'customer_id': range(1, 101),
    'name':  [f'Customer_{i}' for i in range(1, 101)],
    'tier':  np.random.choice(['bronze','silver','gold'], 100),
})
orders = pd.DataFrame({
    'order_id':    range(1, 201),
    'customer_id': np.random.choice(range(1, 115), 200),  # some IDs > 100 = no match
    'amount':      np.random.uniform(20, 500, 200).round(2),
})
products = pd.DataFrame({'product_id': [1,2,3,4,5], 'name': ['Pen','Notebook','Bag','Lamp','Desk']})
product_tags = pd.DataFrame({
    'product_id': [1,1,2,2,3,3,4,5,5],
    'tag':        ['office','writing','office','paper','travel','storage','lighting','furniture','office'],
})

# 1. Left join with indicator
# YOUR CODE HERE

# 2. Orders with no customer
# YOUR CODE HERE

# 3. Many-to-many: products x product_tags
# YOUR CODE HERE

# 4. Count products per tag
# YOUR CODE HERE

# 5. validate many_to_one (will raise if orders has duplicate customer IDs... use customers join)
# YOUR CODE HERE

---
## Exercise 26 — pd.wide_to_long()

**Scenario:** Survey data has repeated column patterns (score_2021, score_2022, rank_2021, rank_2022).

Tasks:
1. Use `pd.wide_to_long()` to unpivot columns named `score_YYYY` and `rank_YYYY` into long format
2. The result should have columns: `student_id`, `name`, `year`, `score`, `rank`
3. Sort by `student_id`, then `year`
4. Compute each student's average score across years
5. Find students who improved their score from 2021 to 2023

In [ ]:
np.random.seed(26)
n = 50
students = pd.DataFrame({
    'student_id': range(1, n+1),
    'name': [f'Student_{i}' for i in range(1, n+1)],
    'score_2021': np.random.randint(50, 100, n),
    'score_2022': np.random.randint(50, 100, n),
    'score_2023': np.random.randint(50, 100, n),
    'rank_2021':  np.random.randint(1, n+1, n),
    'rank_2022':  np.random.randint(1, n+1, n),
    'rank_2023':  np.random.randint(1, n+1, n),
})
print('Wide shape:', students.shape)
print(students.head(2))

# 1 & 2. wide_to_long
# Hint: stubnames=['score','rank'], i=['student_id','name'], j='year', sep='_'
# YOUR CODE HERE

# 3. Sort
# YOUR CODE HERE

# 4. Average score per student
# YOUR CODE HERE

# 5. Students who improved 2021 → 2023
# YOUR CODE HERE

---
## Exercise 27 — Advanced String Operations

**Scenario:** Parse and extract structured data from messy raw text fields.

Tasks:
1. Use `.str.extract()` to pull the **area code** from phone numbers like `(212) 555-1234`
2. Use `.str.findall()` to extract all **hashtags** from tweet text
3. Use `.str.replace()` with a regex to normalise whitespace (multiple spaces → single space)
4. Use `.str.split(expand=True)` to split `full_address` into `street`, `city`, `state`
5. Count the number of words in each `bio` field using `.str.count()`

In [ ]:
contacts = pd.DataFrame({
    'name':  ['Alice Smith', 'Bob Jones', 'Carol Lee', 'Dave Brown', 'Eve Taylor'],
    'phone': ['(212) 555-1234', '(415) 555-9876', '(312) 555-4567', '(646) 555-0011', '(310) 555-7788'],
    'tweet': [
        'Loving #pandas and #python today!',
        '#datascience is the future #ai',
        'No hashtags here.',
        '#machinelearning #deeplearning rocks',
        'Check out #pandas #numpy #scipy',
    ],
    'full_address': [
        '123 Main St, New York, NY',
        '456 Oak Ave, San Francisco, CA',
        '789 Pine Rd, Chicago, IL',
        '321 Elm St, Brooklyn, NY',
        '654 Maple Dr, Los Angeles, CA',
    ],
    'bio': [
        'Data  scientist  at  BigCorp',
        'ML   engineer and blogger',
        'Software developer',
        'Analyst at   TechFirm  Inc',
        'Researcher  and  writer',
    ],
})

# 1. Extract area code
# YOUR CODE HERE

# 2. Extract all hashtags
# YOUR CODE HERE

# 3. Normalise whitespace in bio
# YOUR CODE HERE

# 4. Split full_address
# YOUR CODE HERE

# 5. Word count in bio
# YOUR CODE HERE

---
## Exercise 28 — DateTime Advanced

**Scenario:** Process global trading data with time zones and business day calendars.

Tasks:
1. Localise a naive UTC timestamp column using `dt.tz_localize('UTC')`
2. Convert to `'US/Eastern'` timezone using `dt.tz_convert()`
3. Generate a date range of the next **10 business days** from today using `pd.bdate_range()`
4. Use `pd.offsets.BusinessDay(n)` to add 5 business days to each trade date
5. Compute the number of business days between `open_date` and `close_date` using `np.busday_count`

In [ ]:
import pytz

np.random.seed(28)
n = 100
base = pd.Timestamp('2024-01-01 09:30:00')
trades = pd.DataFrame({
    'trade_id':  range(1, n+1),
    'timestamp_utc': [base + pd.Timedelta(minutes=np.random.randint(0, 60*8)) for _ in range(n)],
    'open_date':  pd.to_datetime(np.random.choice(
        pd.date_range('2024-01-01', periods=60).astype(str), n)),
    'close_date': pd.to_datetime(np.random.choice(
        pd.date_range('2024-03-01', periods=60).astype(str), n)),
    'amount':     np.random.uniform(1000, 50000, n).round(2),
})
print(trades.head(3))

# 1. Localize timestamp_utc
# YOUR CODE HERE

# 2. Convert to US/Eastern
# YOUR CODE HERE

# 3. Next 10 business days from 2024-01-01
# YOUR CODE HERE

# 4. Add 5 business days to open_date
# YOUR CODE HERE

# 5. Business days between open_date and close_date
# YOUR CODE HERE

---
## Exercise 29 — Chunked CSV Processing

**Scenario:** Process a large CSV that doesn't fit in memory by reading it in chunks.

Tasks:
1. Write a 200K-row CSV to disk using pandas (simulate a large file)
2. Read it back in chunks of 10K rows using `chunksize`
3. For each chunk, compute total revenue per `region` — accumulate results
4. Combine all chunk results and get the final total revenue per region
5. Verify the chunked result matches reading the whole file at once

In [ ]:
import os, tempfile

np.random.seed(29)
n = 200_000
big_df = pd.DataFrame({
    'order_id': range(n),
    'region':   np.random.choice(['North','South','East','West'], n),
    'product':  np.random.choice(['A','B','C'], n),
    'revenue':  np.random.exponential(100, n).round(2),
    'qty':      np.random.randint(1, 20, n),
})
tmp_path = os.path.join(tempfile.gettempdir(), 'big_sales.csv')
big_df.to_csv(tmp_path, index=False)
print(f'Wrote {n:,} rows to {tmp_path}')

# 2 & 3. Read in chunks and accumulate
# YOUR CODE HERE
# chunks = []
# for chunk in pd.read_csv(tmp_path, chunksize=10_000):
#     ...

# 4. Combine
# YOUR CODE HERE

# 5. Verify
# YOUR CODE HERE

---
## Exercise 30 — Rolling & Expanding Window Functions (Advanced)

**Scenario:** Build technical indicators for a stock price series.

Tasks:
1. Compute a **20-day Simple Moving Average (SMA)** of the close price
2. Compute a **20-day Exponential Moving Average (EMA)** using `ewm(span=20)`
3. Compute **Bollinger Bands**: upper = SMA + 2*std, lower = SMA - 2*std (20-day window)
4. Compute **RSI (simplified)**: 14-day average gain / (avg gain + avg loss) * 100
5. Use `expanding()` to compute the all-time cumulative max (running peak) of the close price

In [ ]:
np.random.seed(30)
n = 252  # ~1 trading year
price = pd.DataFrame({'date': pd.bdate_range('2024-01-01', periods=n)})
price['close'] = 100 * np.exp(np.cumsum(np.random.normal(0.0005, 0.015, n)))
price = price.set_index('date')
print(price.head())

# 1. 20-day SMA
# YOUR CODE HERE

# 2. 20-day EMA
# YOUR CODE HERE

# 3. Bollinger Bands
# YOUR CODE HERE

# 4. RSI (14-day)
delta = price['close'].diff()
# YOUR CODE HERE — compute gain, loss, RSI

# 5. Expanding cumulative max (all-time high)
# YOUR CODE HERE

print(price[['close','sma20','ema20','bb_upper','bb_lower','rsi14','all_time_high']].tail())

---
## Exercise 31 — groupby + transform (Broadcast Aggregation)

**Scenario:** Enrich each row with group-level statistics without losing any rows.

Tasks:
1. Add `dept_avg_salary` — the average salary for each employee's department (same shape as original)
2. Add `dept_rank` — rank of each employee's salary within their department
3. Add `salary_vs_dept_avg` — how much above/below the dept average each employee is (%)
4. Add `dept_headcount` — number of employees in each department
5. Flag employees whose salary is in the top 25% of their department

In [ ]:
np.random.seed(31)
n = 500
employees = pd.DataFrame({
    'emp_id': range(1, n+1),
    'dept':   np.random.choice(['Eng','Sales','HR','Finance','Ops'], n),
    'level':  np.random.choice(['Junior','Mid','Senior'], n, p=[0.4,0.4,0.2]),
    'salary': np.random.normal(70000, 20000, n).clip(30000, 200000).round(-2),
})
print(employees.head())

# 1. dept_avg_salary via transform
# YOUR CODE HERE

# 2. dept_rank (1 = highest salary in dept)
# YOUR CODE HERE

# 3. salary_vs_dept_avg %
# YOUR CODE HERE

# 4. dept_headcount
# YOUR CODE HERE

# 5. top 25% flag within dept
# YOUR CODE HERE

print(employees[['emp_id','dept','salary','dept_avg_salary','dept_rank','top25_flag']].head(10))

---
## Exercise 32 — apply(axis=1) vs Vectorized Operations

**Scenario:** Compute shipping costs using complex business rules, then optimize.

Tasks:
1. Write a function `calc_shipping(row)` that returns shipping cost based on `weight`, `distance`, and `zone`
   - zone A: base 5 + 0.01 * distance + 0.5 * weight
   - zone B: base 8 + 0.015 * distance + 0.7 * weight
   - zone C: base 12 + 0.02 * distance + 1.0 * weight
2. Apply it with `apply(axis=1)` and time it
3. Implement the same logic using **vectorized** `np.select()` and time it
4. Verify both approaches produce identical results
5. Compute the speedup ratio

In [ ]:
import time
np.random.seed(32)
n = 50_000
shipments = pd.DataFrame({
    'order_id': range(n),
    'weight':   np.random.uniform(0.1, 30, n).round(2),
    'distance': np.random.randint(10, 2000, n),
    'zone':     np.random.choice(['A','B','C'], n),
})

# 1 & 2. apply(axis=1)
def calc_shipping(row):
    if row['zone'] == 'A':
        return 5 + 0.01 * row['distance'] + 0.5 * row['weight']
    elif row['zone'] == 'B':
        return 8 + 0.015 * row['distance'] + 0.7 * row['weight']
    else:
        return 12 + 0.02 * row['distance'] + 1.0 * row['weight']

t0 = time.time()
shipments['cost_apply'] = shipments.apply(calc_shipping, axis=1)
t_apply = time.time() - t0
print(f'apply: {t_apply:.3f}s')

# 3. Vectorized with np.select
t0 = time.time()
# YOUR CODE HERE
t_vec = time.time() - t0
print(f'vectorized: {t_vec:.3f}s')

# 4 & 5. Verify + speedup
# YOUR CODE HERE

---
## Exercise 33 — Hierarchical GroupBy (Multiple Levels)

**Scenario:** Drill down through country → region → city in an e-commerce dataset.

Tasks:
1. GroupBy `['country', 'region']` → sum revenue
2. GroupBy `['country', 'region', 'city']` → count orders and mean order value
3. Use `unstack()` on a two-level groupby result to create a pivot-like view
4. Find the **best city** (highest total revenue) per country
5. Compute revenue as a **percentage of its country total** using `groupby + transform`

In [ ]:
np.random.seed(33)
n = 3000
geo = pd.DataFrame({
    'order_id':    range(n),
    'country':     np.random.choice(['US','UK','DE'], n, p=[0.5,0.3,0.2]),
    'region':      np.random.choice(['North','South','East','West'], n),
    'city':        np.random.choice(['CityA','CityB','CityC','CityD'], n),
    'category':    np.random.choice(['Electronics','Clothing','Food'], n),
    'revenue':     np.random.exponential(150, n).round(2),
    'order_value': np.random.uniform(20, 500, n).round(2),
})

# 1. Country + Region → revenue sum
# YOUR CODE HERE

# 2. Country + Region + City → order count + mean order value
# YOUR CODE HERE

# 3. Unstack category level to see wide view (country x category)
# YOUR CODE HERE

# 4. Best city per country by total revenue
# YOUR CODE HERE

# 5. Revenue % of country total per row
# YOUR CODE HERE

---
## Exercise 34 — merge_asof() — Time-Based Join

**Scenario:** Join trade execution data with the most recent bid/ask quote.

Tasks:
1. Use `pd.merge_asof()` to join each `trade` with the most recent `quote` at or before the trade time
2. Compute the **spread** at trade time: `ask - bid`
3. Check whether each trade was executed at the bid, ask, or in-between
4. Try `direction='forward'` to get the next quote after each trade instead
5. Count how many trades had no prior quote available (result is NaN)

In [ ]:
np.random.seed(34)
quote_times = pd.date_range('2024-01-15 09:30:00', periods=100, freq='30s')
quotes = pd.DataFrame({
    'time': quote_times,
    'bid':  (150 + np.cumsum(np.random.normal(0, 0.05, 100))).round(3),
})
quotes['ask'] = (quotes['bid'] + np.random.uniform(0.01, 0.05, 100)).round(3)

trade_times = pd.date_range('2024-01-15 09:30:00', periods=30, freq='90s')
trades = pd.DataFrame({
    'time':  trade_times,
    'price': (150 + np.cumsum(np.random.normal(0, 0.05, 30))).round(3),
    'size':  np.random.randint(100, 1000, 30),
})
# Both must be sorted by 'time'
quotes = quotes.sort_values('time')
trades = trades.sort_values('time')

# 1. merge_asof (backward = default)
# YOUR CODE HERE

# 2. Spread at trade time
# YOUR CODE HERE

# 3. Classify trade vs quote
# YOUR CODE HERE

# 4. direction='forward'
# YOUR CODE HERE

# 5. Count NaN bid rows
# YOUR CODE HERE

---
## Exercise 35 — pivot_table() with margins & Multiple aggfunc

**Scenario:** Build an executive summary table of sales across channels and regions.

Tasks:
1. Create a pivot table: rows=`region`, columns=`channel`, values=`revenue`, aggfunc=`sum`
2. Add `margins=True` to get row and column totals
3. Create a pivot with **two aggfuncs**: `{'revenue': ['sum','mean'], 'units': 'sum'}`
4. Fill missing combinations with 0 using `fill_value=0`
5. Use `observed=True` with a categorical column to suppress empty categories

In [ ]:
np.random.seed(35)
n = 2000
sales = pd.DataFrame({
    'region':  np.random.choice(['North','South','East','West'], n),
    'channel': np.random.choice(['Online','Retail','Wholesale'], n, p=[0.5,0.3,0.2]),
    'product': np.random.choice(['A','B','C','D'], n),
    'revenue': np.random.exponential(300, n).round(2),
    'units':   np.random.randint(1, 50, n),
})

# 1. Basic pivot
# YOUR CODE HERE

# 2. With margins
# YOUR CODE HERE

# 3. Multiple aggfuncs
# YOUR CODE HERE

# 4. fill_value=0
# YOUR CODE HERE

# 5. Categorical + observed=True
sales['channel_cat'] = pd.Categorical(sales['channel'],
    categories=['Online','Retail','Wholesale','Direct'], ordered=False)
# YOUR CODE HERE

---
## Exercise 36 — DataFrame.compare()

**Scenario:** Audit changes made to a customer database between two versions.

Tasks:
1. Use `df1.compare(df2)` to find all cells that changed
2. Use `result_names=('before', 'after')` for clearer column labels
3. Count how many rows changed (have at least one difference)
4. Find which columns had the most changes
5. Extract only the rows where `email` changed

In [ ]:
np.random.seed(36)
n = 100
base = pd.DataFrame({
    'customer_id': range(1, n+1),
    'name':   [f'Customer_{i}' for i in range(1, n+1)],
    'email':  [f'user{i}@email.com' for i in range(1, n+1)],
    'score':  np.random.randint(1, 100, n),
    'tier':   np.random.choice(['free','paid','premium'], n),
}).set_index('customer_id')

# Simulate some changes
updated = base.copy()
idx_score  = np.random.choice(n, 15, replace=False)
idx_tier   = np.random.choice(n, 10, replace=False)
idx_email  = np.random.choice(n, 5,  replace=False)
updated.iloc[idx_score, updated.columns.get_loc('score')] += 10
updated.iloc[idx_tier,  updated.columns.get_loc('tier')]  = 'premium'
for i in idx_email:
    updated.iloc[i, updated.columns.get_loc('email')] = f'new{i}@updated.com'

# 1. compare
# YOUR CODE HERE

# 2. result_names
# YOUR CODE HERE

# 3. Count changed rows
# YOUR CODE HERE

# 4. Changes per column
# YOUR CODE HERE

# 5. Rows where email changed
# YOUR CODE HERE

---
## Exercise 37 — pd.json_normalize() — Flatten Nested JSON

**Scenario:** API response returns deeply nested JSON. Flatten it for analysis.

Tasks:
1. Use `pd.json_normalize()` to flatten a list of nested user records
2. Rename columns to replace dots with underscores for easier access
3. Flatten the nested `orders` list within each record using `record_path`
4. Set `max_level=1` to partially flatten (only one level deep)
5. Compute total order value per user from the flattened data

In [ ]:
import json

raw_data = [
    {'id': 1, 'name': 'Alice', 'address': {'city': 'NYC', 'country': 'US'},
     'profile': {'age': 30, 'tier': 'gold'},
     'orders': [{'oid': 101, 'value': 250.0}, {'oid': 102, 'value': 80.0}]},
    {'id': 2, 'name': 'Bob',   'address': {'city': 'LA',  'country': 'US'},
     'profile': {'age': 25, 'tier': 'silver'},
     'orders': [{'oid': 103, 'value': 120.0}]},
    {'id': 3, 'name': 'Carol', 'address': {'city': 'London', 'country': 'UK'},
     'profile': {'age': 35, 'tier': 'bronze'},
     'orders': [{'oid': 104, 'value': 330.0}, {'oid': 105, 'value': 60.0}, {'oid': 106, 'value': 90.0}]},
]

# 1. Basic json_normalize
# YOUR CODE HERE

# 2. Rename dots → underscores
# YOUR CODE HERE

# 3. Flatten orders using record_path
# YOUR CODE HERE

# 4. max_level=1
# YOUR CODE HERE

# 5. Total order value per user
# YOUR CODE HERE

---
## Exercise 38 — IntervalIndex & pd.cut Advanced

**Scenario:** Build custom price bands for a retail product catalogue.

Tasks:
1. Use `pd.cut()` with custom bins and labels to create `price_band`
2. Use `pd.IntervalIndex.from_tuples()` to create a custom index for lookup
3. Map each product's price to a discount tier using the IntervalIndex
4. Use `pd.qcut()` to split prices into equal-frequency quintiles (5 groups)
5. Compare the distribution of products across `cut` bands vs `qcut` quintiles

In [ ]:
np.random.seed(38)
n = 500
products = pd.DataFrame({
    'product_id': range(1, n+1),
    'category':   np.random.choice(['Electronics','Clothing','Food','Sports'], n),
    'price':      np.random.lognormal(4, 1, n).round(2).clip(1, 2000),
})
print(products['price'].describe())

# 1. Custom cut with labels
bins   = [0, 25, 50, 100, 250, 500, np.inf]
labels = ['budget','economy','mid','upper-mid','premium','luxury']
# YOUR CODE HERE

# 2. IntervalIndex for discount tiers
intervals = pd.IntervalIndex.from_tuples([(0,25),(25,100),(100,500),(500,np.inf)], closed='left')
discount_tiers = pd.Series([20, 15, 10, 5], index=intervals, name='discount_pct')
# YOUR CODE HERE — map each product to its discount

# 3. Already done via IntervalIndex lookup — verify
print(products[['price','price_band','discount_pct']].head(10))

# 4. qcut into 5 quintiles
# YOUR CODE HERE

# 5. Compare distributions
# YOUR CODE HERE

---
## Exercise 39 — GroupBy + Cumulative Statistics

**Scenario:** Track running performance metrics for each sales rep over time.

Tasks:
1. Compute `cumsum` of `revenue` per `rep_id` (running total per rep)
2. Compute `cummax` of `revenue` per rep (personal best)
3. Compute `cumprod` of daily `growth_rate` per rep (compound growth)
4. Compute the **cumulative rank** of each rep's daily revenue within their group
5. Compute a **30-day rolling average** within each rep group using `groupby + rolling`

In [ ]:
np.random.seed(39)
n_reps = 5
n_days = 90
rep_ids   = [f'REP{i:02d}' for i in range(1, n_reps+1)]
dates     = pd.date_range('2024-01-01', periods=n_days, freq='D')
idx       = pd.MultiIndex.from_product([rep_ids, dates], names=['rep_id','date'])
perf = pd.DataFrame({
    'revenue':     np.random.exponential(1000, len(idx)).round(2),
    'growth_rate': np.random.uniform(0.98, 1.04, len(idx)).round(4),
}, index=idx).reset_index()
print(perf.head())

# 1. Cumulative revenue per rep
# YOUR CODE HERE

# 2. Personal best revenue
# YOUR CODE HERE

# 3. Compound growth
# YOUR CODE HERE

# 4. Cumulative rank within group
# YOUR CODE HERE

# 5. 30-day rolling avg within rep group
# YOUR CODE HERE

print(perf.tail(10))

---
## Exercise 40 — Mini Project: End-to-End Sales Pipeline

**Scenario:** Build a complete ETL + analysis pipeline for a multi-channel e-commerce company.

**Step 1 — Ingest & Validate:**
- Load orders, customers, and products tables
- Check for duplicates, nulls, and data type issues

**Step 2 — Clean & Enrich:**
- Merge all three tables
- Add `revenue = price * quantity * (1 - discount)`
- Add `age_group` and `customer_tier` columns

**Step 3 — Analyse:**
- Monthly revenue trend (with MoM %)
- Top 5 products by total revenue
- Revenue breakdown by channel and region
- Customer cohort: first-purchase month vs LTV

**Step 4 — Summarise:**
- Print an executive summary: total revenue, orders, AOV, top region, top product
- Export the enriched DataFrame to CSV

In [ ]:
from datetime import timedelta
import os, tempfile

np.random.seed(40)
N_CUSTOMERS = 2000
N_PRODUCTS  = 50
N_ORDERS    = 15000

# ── Raw Tables ──────────────────────────────────────────────
customers_df = pd.DataFrame({
    'customer_id': range(1, N_CUSTOMERS+1),
    'name':        [f'Customer_{i}' for i in range(1, N_CUSTOMERS+1)],
    'age':         np.random.randint(18, 75, N_CUSTOMERS),
    'gender':      np.random.choice(['M','F'], N_CUSTOMERS),
    'region':      np.random.choice(['North','South','East','West'], N_CUSTOMERS),
    'channel':     np.random.choice(['Online','Retail','Mobile'], N_CUSTOMERS, p=[0.5,0.3,0.2]),
})

products_df = pd.DataFrame({
    'product_id':  range(1, N_PRODUCTS+1),
    'product_name':[f'Product_{i}' for i in range(1, N_PRODUCTS+1)],
    'category':    np.random.choice(['Electronics','Clothing','Food','Sports','Home'], N_PRODUCTS),
    'price':       np.random.uniform(5, 500, N_PRODUCTS).round(2),
})

orders_df = pd.DataFrame({
    'order_id':    range(1, N_ORDERS+1),
    'customer_id': np.random.randint(1, N_CUSTOMERS+1, N_ORDERS),
    'product_id':  np.random.randint(1, N_PRODUCTS+1, N_ORDERS),
    'order_date':  pd.to_datetime(np.random.choice(
        pd.date_range('2023-01-01','2024-12-31').astype(str), N_ORDERS)),
    'quantity':    np.random.randint(1, 10, N_ORDERS),
    'discount':    np.random.choice([0, 0.05, 0.10, 0.15, 0.20], N_ORDERS, p=[0.5,0.2,0.15,0.1,0.05]),
})

# ── Step 1: Validate ────────────────────────────────────────
# Check for duplicates and nulls in each table
# YOUR CODE HERE

# ── Step 2: Clean & Enrich ──────────────────────────────────
# Merge orders with customers and products
# Add revenue = price * quantity * (1 - discount)
# Add age_group using pd.cut
# YOUR CODE HERE

# ── Step 3: Analyse ─────────────────────────────────────────
# 3a. Monthly revenue + MoM %
# YOUR CODE HERE

# 3b. Top 5 products by revenue
# YOUR CODE HERE

# 3c. Revenue by channel and region (pivot)
# YOUR CODE HERE

# 3d. First-purchase cohort: cohort month vs avg LTV
# YOUR CODE HERE

# ── Step 4: Executive Summary ───────────────────────────────
# YOUR CODE HERE

# Export enriched df to CSV
# YOUR CODE HERE

---
# ✅ Solutions (21–40)

> Try first!

## Solution 21 — Time Series Resampling

In [ ]:
idx = pd.date_range('2024-01-01', periods=60*24, freq='h')
np.random.seed(7)
sensors = pd.DataFrame({
    'temperature': 20 + 10*np.sin(np.linspace(0, 60*2*np.pi/24, len(idx))) + np.random.normal(0, 2, len(idx)),
    'rainfall':    np.random.exponential(0.5, len(idx)).round(2),
}, index=idx)

daily = sensors.resample('D').agg({'temperature': 'mean', 'rainfall': 'sum'})
print('Daily (head):'); print(daily.head())

weekly_max = sensors['temperature'].resample('W').max()
print('Weekly max temp (head):'); print(weekly_max.head())

monthly = sensors['temperature'].resample('ME').agg(['mean','std']).round(2)
print('Monthly stats:'); print(monthly)

daily_freq = daily['temperature'].asfreq('D')
print('NaN count after asfreq:', daily_freq.isna().sum())  # should be 0 here

sparse = sensors['temperature'].resample('D').mean()
sparse_missing = sparse.copy(); sparse_missing.iloc[3:6] = np.nan
ffill_result = sparse_missing.ffill()
interp_result = sparse_missing.interpolate(method='linear')
print('ffill vs interpolate at gap:'); print(pd.concat([sparse_missing, ffill_result, interp_result], axis=1, keys=['original','ffill','interp']).iloc[2:8])

## Solution 22 — crosstab()

In [ ]:
np.random.seed(21)
n = 1000
hr = pd.DataFrame({'employee_id':range(n),'dept':np.random.choice(['Eng','Sales','HR','Finance','Ops'],n),'gender':np.random.choice(['M','F'],n,p=[0.55,0.45]),'level':np.random.choice(['Junior','Mid','Senior'],n,p=[0.4,0.4,0.2]),'promoted':np.random.choice([0,1],n,p=[0.75,0.25])})

ct = pd.crosstab(hr['dept'], hr['promoted'])
print(ct)

ct_norm = pd.crosstab(hr['dept'], hr['promoted'], normalize='index').round(3)
print(ct_norm)

ct_margins = pd.crosstab(hr['dept'], hr['promoted'], margins=True)
print(ct_margins)

ct_val = pd.crosstab(hr['dept'], hr['gender'], values=hr['promoted'], aggfunc='mean').round(3)
print(ct_val)

best_female_dept = ct_val['F'].idxmax()
print('Best dept for female promotion:', best_female_dept)

## Solution 23 — Custom Aggregation

In [ ]:
np.random.seed(11)
n = 2000
sales = pd.DataFrame({'region':np.random.choice(['North','South','East','West'],n),'product':np.random.choice(['A','B','C','D'],n),'revenue':np.random.exponential(500,n).round(2),'quantity':np.random.randint(1,50,n),'discount':np.random.uniform(0,0.4,n).round(3)})

r1 = sales.groupby('region').agg({'revenue':['sum','mean','max'],'quantity':['sum','mean']})
print(r1)

r2 = sales.groupby('region').agg(
    rev_total=('revenue','sum'), rev_mean=('revenue','mean'), rev_max=('revenue','max'),
    qty_total=('quantity','sum'), qty_mean=('quantity','mean')
).round(2)
print(r2)

def iqr(s): return s.quantile(0.75) - s.quantile(0.25)
print(sales.groupby('region')['revenue'].agg(iqr).round(2))

print(sales.groupby('region')['revenue'].agg(['sum','mean','std', iqr]).round(2))

cv = sales.groupby('region')['revenue'].agg(lambda s: s.std()/s.mean()*100).round(2)
print('Coefficient of variation:'); print(cv)

## Solution 24 — where() & mask()

In [ ]:
np.random.seed(24)
n = 500
df = pd.DataFrame({'temperature':np.random.normal(20,30,n).round(1),'revenue':np.random.normal(500,200,n).round(2),'price':np.random.uniform(10,200,n).round(2)})

df['temp_clean'] = df['temperature'].where((df['temperature'] >= -10) & (df['temperature'] <= 50))
print('Temp NaNs after where:', df['temp_clean'].isna().sum())

df['revenue_clean'] = df['revenue'].mask(df['revenue'] < 0, 0)
print('Revenue negatives after mask:', (df['revenue_clean'] < 0).sum())

cap99 = df['revenue'].quantile(0.99)
df['revenue_capped'] = df['revenue'].where(df['revenue'] <= cap99, cap99)
print(f'Capped at {cap99:.2f} — max now: {df["revenue_capped"].max():.2f}')

df['price_tier'] = np.where(df['price'] > 100, 'high', 'low')
print(df['price_tier'].value_counts())

p1, p99 = df['revenue'].quantile(0.01), df['revenue'].quantile(0.99)
df['rev_trimmed'] = df['revenue'].mask(df['revenue'] < p1).mask(df['revenue'] > p99)
print('Trimmed NaN count:', df['rev_trimmed'].isna().sum())

## Solution 25 — merge() indicator & many-to-many

In [ ]:
np.random.seed(25)
customers = pd.DataFrame({'customer_id':range(1,101),'name':[f'Customer_{i}' for i in range(1,101)],'tier':np.random.choice(['bronze','silver','gold'],100)})
orders = pd.DataFrame({'order_id':range(1,201),'customer_id':np.random.choice(range(1,115),200),'amount':np.random.uniform(20,500,200).round(2)})
products = pd.DataFrame({'product_id':[1,2,3,4,5],'name':['Pen','Notebook','Bag','Lamp','Desk']})
product_tags = pd.DataFrame({'product_id':[1,1,2,2,3,3,4,5,5],'tag':['office','writing','office','paper','travel','storage','lighting','furniture','office']})

merged = orders.merge(customers, on='customer_id', how='left', indicator=True)
print('_merge counts:'); print(merged['_merge'].value_counts())

no_cust = merged[merged['_merge'] == 'left_only']
print('Orders with no customer:', len(no_cust))

prod_tagged = products.merge(product_tags, on='product_id')
print(prod_tagged)

tag_counts = prod_tagged.groupby('tag')['product_id'].count().sort_values(ascending=False)
print(tag_counts)

try:
    orders.merge(customers, on='customer_id', how='inner', validate='many_to_one')
    print('Validation passed')
except Exception as e:
    print('Validation error:', e)

## Solution 26 — wide_to_long()

In [ ]:
np.random.seed(26)
n = 50
students = pd.DataFrame({'student_id':range(1,n+1),'name':[f'Student_{i}' for i in range(1,n+1)],'score_2021':np.random.randint(50,100,n),'score_2022':np.random.randint(50,100,n),'score_2023':np.random.randint(50,100,n),'rank_2021':np.random.randint(1,n+1,n),'rank_2022':np.random.randint(1,n+1,n),'rank_2023':np.random.randint(1,n+1,n)})

long = pd.wide_to_long(students, stubnames=['score','rank'], i=['student_id','name'], j='year', sep='_')
long = long.reset_index().sort_values(['student_id','year'])
print('Long shape:', long.shape); print(long.head(6))

avg_score = long.groupby('student_id')['score'].mean().round(2)
print(avg_score.head())

pivot_s = long.pivot(index='student_id', columns='year', values='score')
improved = pivot_s[pivot_s[2023] > pivot_s[2021]]
print(f'Students who improved 2021->2023: {len(improved)}')

## Solution 27 — Advanced String Ops

In [ ]:
contacts = pd.DataFrame({'name':['Alice Smith','Bob Jones','Carol Lee','Dave Brown','Eve Taylor'],'phone':['(212) 555-1234','(415) 555-9876','(312) 555-4567','(646) 555-0011','(310) 555-7788'],'tweet':['Loving #pandas and #python today!','#datascience is the future #ai','No hashtags here.','#machinelearning #deeplearning rocks','Check out #pandas #numpy #scipy'],'full_address':['123 Main St, New York, NY','456 Oak Ave, San Francisco, CA','789 Pine Rd, Chicago, IL','321 Elm St, Brooklyn, NY','654 Maple Dr, Los Angeles, CA'],'bio':['Data  scientist  at  BigCorp','ML   engineer and blogger','Software developer','Analyst at   TechFirm  Inc','Researcher  and  writer']})

contacts['area_code'] = contacts['phone'].str.extract(r'\((\d{3})\)')
print(contacts[['name','area_code']])

contacts['hashtags'] = contacts['tweet'].str.findall(r'#\w+')
print(contacts[['name','hashtags']])

contacts['bio_clean'] = contacts['bio'].str.replace(r'\s+', ' ', regex=True).str.strip()
print(contacts[['bio','bio_clean']])

addr_split = contacts['full_address'].str.split(', ', expand=True)
addr_split.columns = ['street','city','state']
print(addr_split)

contacts['word_count'] = contacts['bio_clean'].str.count(r'\S+')
print(contacts[['name','word_count']])

## Solution 28 — DateTime Advanced

In [ ]:
import pytz
np.random.seed(28)
n = 100
base = pd.Timestamp('2024-01-15 09:30:00')
trades = pd.DataFrame({'trade_id':range(1,n+1),'timestamp_utc':[base+pd.Timedelta(minutes=int(x)) for x in np.random.randint(0,60*8,n)],'open_date':pd.to_datetime(np.random.choice(pd.date_range('2024-01-01',periods=60).astype(str),n)),'close_date':pd.to_datetime(np.random.choice(pd.date_range('2024-03-01',periods=60).astype(str),n)),'amount':np.random.uniform(1000,50000,n).round(2)})

trades['ts_utc'] = trades['timestamp_utc'].dt.tz_localize('UTC')
print(trades['ts_utc'].head(2))

trades['ts_eastern'] = trades['ts_utc'].dt.tz_convert('US/Eastern')
print(trades['ts_eastern'].head(2))

bdays = pd.bdate_range('2024-01-01', periods=10)
print('Next 10 business days:', bdays.tolist())

trades['settlement'] = trades['open_date'] + pd.offsets.BusinessDay(5)
print(trades[['open_date','settlement']].head())

od = trades['open_date'].dt.date.astype(str).values
cd = trades['close_date'].dt.date.astype(str).values
trades['bdays_open'] = np.busday_count(od, cd)
print(trades[['open_date','close_date','bdays_open']].head())

## Solution 29 — Chunked CSV Processing

In [ ]:
import os, tempfile
np.random.seed(29)
n = 200_000
big_df = pd.DataFrame({'order_id':range(n),'region':np.random.choice(['North','South','East','West'],n),'product':np.random.choice(['A','B','C'],n),'revenue':np.random.exponential(100,n).round(2),'qty':np.random.randint(1,20,n)})
tmp_path = os.path.join(tempfile.gettempdir(), 'big_sales.csv')
big_df.to_csv(tmp_path, index=False)

chunks = []
for chunk in pd.read_csv(tmp_path, chunksize=10_000):
    chunks.append(chunk.groupby('region')['revenue'].sum())

chunked_result = pd.concat(chunks).groupby(level=0).sum().round(2)
print('Chunked result:'); print(chunked_result)

full_result = big_df.groupby('region')['revenue'].sum().round(2)
print('Full result:');    print(full_result)
print('Match:', chunked_result.equals(full_result))

## Solution 30 — Rolling & Expanding Advanced

In [ ]:
np.random.seed(30)
n = 252
price = pd.DataFrame({'close': 100 * np.exp(np.cumsum(np.random.normal(0.0005, 0.015, n)))},
                     index=pd.bdate_range('2024-01-01', periods=n))

price['sma20'] = price['close'].rolling(20).mean()
price['ema20'] = price['close'].ewm(span=20, adjust=False).mean()

std20 = price['close'].rolling(20).std()
price['bb_upper'] = price['sma20'] + 2 * std20
price['bb_lower'] = price['sma20'] - 2 * std20

delta = price['close'].diff()
gain  = delta.clip(lower=0)
loss  = (-delta).clip(lower=0)
avg_gain = gain.rolling(14).mean()
avg_loss = loss.rolling(14).mean()
rs = avg_gain / avg_loss
price['rsi14'] = (100 - (100 / (1 + rs))).round(2)

price['all_time_high'] = price['close'].expanding().max()
print(price[['close','sma20','ema20','bb_upper','bb_lower','rsi14','all_time_high']].tail())

## Solution 31 — groupby + transform

In [ ]:
np.random.seed(31)
n = 500
employees = pd.DataFrame({'emp_id':range(1,n+1),'dept':np.random.choice(['Eng','Sales','HR','Finance','Ops'],n),'level':np.random.choice(['Junior','Mid','Senior'],n,p=[0.4,0.4,0.2]),'salary':np.random.normal(70000,20000,n).clip(30000,200000).round(-2)})

employees['dept_avg_salary'] = employees.groupby('dept')['salary'].transform('mean').round(2)
employees['dept_rank'] = employees.groupby('dept')['salary'].rank(ascending=False, method='min').astype(int)
employees['salary_vs_dept_avg'] = ((employees['salary'] - employees['dept_avg_salary']) / employees['dept_avg_salary'] * 100).round(2)
employees['dept_headcount'] = employees.groupby('dept')['emp_id'].transform('count')
q75 = employees.groupby('dept')['salary'].transform(lambda x: x.quantile(0.75))
employees['top25_flag'] = employees['salary'] >= q75
print(employees[['emp_id','dept','salary','dept_avg_salary','dept_rank','top25_flag']].head(10))

## Solution 32 — apply(axis=1) vs vectorized

In [ ]:
import time
np.random.seed(32)
n = 50_000
shipments = pd.DataFrame({'order_id':range(n),'weight':np.random.uniform(0.1,30,n).round(2),'distance':np.random.randint(10,2000,n),'zone':np.random.choice(['A','B','C'],n)})

def calc_shipping(row):
    if row['zone']=='A': return 5  + 0.01 *row['distance'] + 0.5*row['weight']
    elif row['zone']=='B': return 8 + 0.015*row['distance'] + 0.7*row['weight']
    else: return 12 + 0.02*row['distance'] + 1.0*row['weight']

t0 = time.time()
shipments['cost_apply'] = shipments.apply(calc_shipping, axis=1)
t_apply = time.time() - t0
print(f'apply: {t_apply:.3f}s')

t0 = time.time()
conds = [shipments['zone']=='A', shipments['zone']=='B', shipments['zone']=='C']
choices = [
    5  + 0.01 *shipments['distance'] + 0.5*shipments['weight'],
    8  + 0.015*shipments['distance'] + 0.7*shipments['weight'],
    12 + 0.02 *shipments['distance'] + 1.0*shipments['weight'],
]
shipments['cost_vec'] = np.select(conds, choices)
t_vec = time.time() - t0
print(f'vectorized: {t_vec:.3f}s')

match = np.allclose(shipments['cost_apply'], shipments['cost_vec'])
print(f'Results match: {match} | Speedup: {t_apply/t_vec:.1f}x')

## Solution 33 — Hierarchical GroupBy

In [ ]:
np.random.seed(33)
n = 3000
geo = pd.DataFrame({'order_id':range(n),'country':np.random.choice(['US','UK','DE'],n,p=[0.5,0.3,0.2]),'region':np.random.choice(['North','South','East','West'],n),'city':np.random.choice(['CityA','CityB','CityC','CityD'],n),'category':np.random.choice(['Electronics','Clothing','Food'],n),'revenue':np.random.exponential(150,n).round(2),'order_value':np.random.uniform(20,500,n).round(2)})

r1 = geo.groupby(['country','region'])['revenue'].sum().round(2)
print(r1.head(8))

r2 = geo.groupby(['country','region','city']).agg(orders=('order_id','count'), mean_val=('order_value','mean')).round(2)
print(r2.head(6))

cat_pivot = geo.groupby(['country','category'])['revenue'].sum().unstack(fill_value=0).round(0)
print(cat_pivot)

city_rev = geo.groupby(['country','city'])['revenue'].sum()
best_city = city_rev.groupby(level='country').idxmax().apply(lambda x: x[1])
print('Best city per country:', best_city)

geo['country_total'] = geo.groupby('country')['revenue'].transform('sum')
geo['rev_pct'] = (geo['revenue'] / geo['country_total'] * 100).round(4)
print(geo[['country','revenue','rev_pct']].head())

## Solution 34 — merge_asof()

In [ ]:
np.random.seed(34)
quote_times = pd.date_range('2024-01-15 09:30:00', periods=100, freq='30s')
quotes = pd.DataFrame({'time':quote_times,'bid':(150+np.cumsum(np.random.normal(0,0.05,100))).round(3)})
quotes['ask'] = (quotes['bid']+np.random.uniform(0.01,0.05,100)).round(3)
trade_times = pd.date_range('2024-01-15 09:30:00', periods=30, freq='90s')
trades = pd.DataFrame({'time':trade_times,'price':(150+np.cumsum(np.random.normal(0,0.05,30))).round(3),'size':np.random.randint(100,1000,30)})
quotes = quotes.sort_values('time'); trades = trades.sort_values('time')

joined = pd.merge_asof(trades, quotes, on='time', direction='backward')
print(joined.head())

joined['spread'] = (joined['ask'] - joined['bid']).round(4)
print('Avg spread at trade time:', joined['spread'].mean().round(4))

joined['side'] = np.where(joined['price'] >= joined['ask'], 'buy',
                  np.where(joined['price'] <= joined['bid'], 'sell', 'mid'))
print(joined['side'].value_counts())

fwd = pd.merge_asof(trades, quotes, on='time', direction='forward')
print('Forward join NaN bid:', fwd['bid'].isna().sum())

print('No prior quote:', joined['bid'].isna().sum())

## Solution 35 — pivot_table() advanced

In [ ]:
np.random.seed(35)
n = 2000
sales = pd.DataFrame({'region':np.random.choice(['North','South','East','West'],n),'channel':np.random.choice(['Online','Retail','Wholesale'],n,p=[0.5,0.3,0.2]),'product':np.random.choice(['A','B','C','D'],n),'revenue':np.random.exponential(300,n).round(2),'units':np.random.randint(1,50,n)})

pt1 = pd.pivot_table(sales, values='revenue', index='region', columns='channel', aggfunc='sum')
print(pt1.round(0))

pt2 = pd.pivot_table(sales, values='revenue', index='region', columns='channel', aggfunc='sum', margins=True)
print(pt2.round(0))

pt3 = pd.pivot_table(sales, values={'revenue':['sum','mean'],'units':'sum'}, index='region', columns='channel')
print(pt3.round(0))

pt4 = pd.pivot_table(sales, values='revenue', index='region', columns='channel', aggfunc='sum', fill_value=0)
print(pt4.round(0))

sales['channel_cat'] = pd.Categorical(sales['channel'], categories=['Online','Retail','Wholesale','Direct'], ordered=False)
pt5 = pd.pivot_table(sales, values='revenue', index='region', columns='channel_cat', aggfunc='sum', fill_value=0, observed=True)
print(pt5.round(0))  # 'Direct' column excluded

## Solution 36 — DataFrame.compare()

In [ ]:
np.random.seed(36)
n = 100
base = pd.DataFrame({'customer_id':range(1,n+1),'name':[f'Customer_{i}' for i in range(1,n+1)],'email':[f'user{i}@email.com' for i in range(1,n+1)],'score':np.random.randint(1,100,n),'tier':np.random.choice(['free','paid','premium'],n)}).set_index('customer_id')
updated = base.copy()
idx_score=np.random.choice(n,15,replace=False); idx_tier=np.random.choice(n,10,replace=False); idx_email=np.random.choice(n,5,replace=False)
updated.iloc[idx_score, updated.columns.get_loc('score')] += 10
updated.iloc[idx_tier,  updated.columns.get_loc('tier')]  = 'premium'
for i in idx_email: updated.iloc[i, updated.columns.get_loc('email')] = f'new{i}@updated.com'

diff = base.compare(updated)
print('Changed cells shape:', diff.shape)

diff2 = base.compare(updated, result_names=('before','after'))
print(diff2.head())

changed_rows = diff.index
print('Changed row count:', len(changed_rows))

changes_per_col = diff.notna().any(axis=1).groupby(level=0).sum() if diff.columns.nlevels > 1 else None
col_change_count = diff.notna().sum().groupby(level=0).sum()
print('Changes per column:'); print(col_change_count)

email_diff = diff2[diff2['email'].notna().any(axis=1)] if 'email' in diff2.columns.get_level_values(0) else 'no email changes'
print('Email changes:'); print(email_diff)

## Solution 37 — pd.json_normalize()

In [ ]:
raw_data = [{'id':1,'name':'Alice','address':{'city':'NYC','country':'US'},'profile':{'age':30,'tier':'gold'},'orders':[{'oid':101,'value':250.0},{'oid':102,'value':80.0}]},{'id':2,'name':'Bob','address':{'city':'LA','country':'US'},'profile':{'age':25,'tier':'silver'},'orders':[{'oid':103,'value':120.0}]},{'id':3,'name':'Carol','address':{'city':'London','country':'UK'},'profile':{'age':35,'tier':'bronze'},'orders':[{'oid':104,'value':330.0},{'oid':105,'value':60.0},{'oid':106,'value':90.0}]}]

flat = pd.json_normalize(raw_data)
print('Columns:', flat.columns.tolist())
print(flat[['id','name','address.city','profile.tier']])

flat.columns = flat.columns.str.replace('.', '_', regex=False)
print('Renamed:', flat.columns.tolist())

orders_flat = pd.json_normalize(raw_data, record_path='orders', meta=['id','name'])
print(orders_flat)

partial = pd.json_normalize(raw_data, max_level=1)
print('max_level=1 cols:', partial.columns.tolist())

total_per_user = orders_flat.groupby('id')['value'].sum()
print('Total order value per user:'); print(total_per_user)

## Solution 38 — IntervalIndex & pd.cut

In [ ]:
np.random.seed(38)
n = 500
products = pd.DataFrame({'product_id':range(1,n+1),'category':np.random.choice(['Electronics','Clothing','Food','Sports'],n),'price':np.random.lognormal(4,1,n).round(2).clip(1,2000)})

bins = [0,25,50,100,250,500,np.inf]
labels = ['budget','economy','mid','upper-mid','premium','luxury']
products['price_band'] = pd.cut(products['price'], bins=bins, labels=labels)
print(products['price_band'].value_counts().sort_index())

intervals = pd.IntervalIndex.from_tuples([(0,25),(25,100),(100,500),(500,np.inf)], closed='left')
discount_tiers = pd.Series([20,15,10,5], index=intervals, name='discount_pct')
products['discount_pct'] = pd.cut(products['price'], bins=intervals).map(discount_tiers)
print(products[['price','price_band','discount_pct']].head(10))

products['price_quintile'] = pd.qcut(products['price'], q=5, labels=['Q1','Q2','Q3','Q4','Q5'])
print('qcut quintiles:'); print(products['price_quintile'].value_counts().sort_index())

comp = pd.crosstab(products['price_band'], products['price_quintile'])
print('cut vs qcut distribution:'); print(comp)

## Solution 39 — GroupBy + Cumulative Stats

In [ ]:
np.random.seed(39)
n_reps=5; n_days=90
rep_ids=[f'REP{i:02d}' for i in range(1,n_reps+1)]
dates=pd.date_range('2024-01-01',periods=n_days,freq='D')
idx=pd.MultiIndex.from_product([rep_ids,dates],names=['rep_id','date'])
perf=pd.DataFrame({'revenue':np.random.exponential(1000,len(idx)).round(2),'growth_rate':np.random.uniform(0.98,1.04,len(idx)).round(4)},index=idx).reset_index()
perf = perf.sort_values(['rep_id','date'])

perf['cum_revenue']   = perf.groupby('rep_id')['revenue'].cumsum()
perf['personal_best'] = perf.groupby('rep_id')['revenue'].cummax()
perf['compound_growth'] = perf.groupby('rep_id')['growth_rate'].cumprod()
perf['cum_rank'] = perf.groupby(['rep_id','date'])['revenue'].rank(pct=True)
perf['rolling_avg_30'] = (
    perf.groupby('rep_id')['revenue']
        .transform(lambda x: x.rolling(30, min_periods=1).mean())
).round(2)
print(perf.tail(10))

## Solution 40 — End-to-End Sales Pipeline

In [ ]:
import os, tempfile
np.random.seed(40)
N_CUSTOMERS=2000; N_PRODUCTS=50; N_ORDERS=15000

customers_df = pd.DataFrame({'customer_id':range(1,N_CUSTOMERS+1),'name':[f'Customer_{i}' for i in range(1,N_CUSTOMERS+1)],'age':np.random.randint(18,75,N_CUSTOMERS),'gender':np.random.choice(['M','F'],N_CUSTOMERS),'region':np.random.choice(['North','South','East','West'],N_CUSTOMERS),'channel':np.random.choice(['Online','Retail','Mobile'],N_CUSTOMERS,p=[0.5,0.3,0.2])})
products_df = pd.DataFrame({'product_id':range(1,N_PRODUCTS+1),'product_name':[f'Product_{i}' for i in range(1,N_PRODUCTS+1)],'category':np.random.choice(['Electronics','Clothing','Food','Sports','Home'],N_PRODUCTS),'price':np.random.uniform(5,500,N_PRODUCTS).round(2)})
orders_df = pd.DataFrame({'order_id':range(1,N_ORDERS+1),'customer_id':np.random.randint(1,N_CUSTOMERS+1,N_ORDERS),'product_id':np.random.randint(1,N_PRODUCTS+1,N_ORDERS),'order_date':pd.to_datetime(np.random.choice(pd.date_range('2023-01-01','2024-12-31').astype(str),N_ORDERS)),'quantity':np.random.randint(1,10,N_ORDERS),'discount':np.random.choice([0,0.05,0.10,0.15,0.20],N_ORDERS,p=[0.5,0.2,0.15,0.1,0.05])})

# Step 1: Validate
for name, df in [('customers',customers_df),('products',products_df),('orders',orders_df)]:
    print(f'{name}: {len(df)} rows | dups={df.duplicated().sum()} | nulls={df.isna().sum().sum()}')

# Step 2: Merge & enrich
enriched = (
    orders_df
    .merge(customers_df, on='customer_id', how='left')
    .merge(products_df,  on='product_id',  how='left')
    .assign(
        revenue   = lambda d: (d['price'] * d['quantity'] * (1 - d['discount'])).round(2),
        age_group = lambda d: pd.cut(d['age'], bins=[17,30,45,65,100], labels=['18-30','31-45','46-65','65+']),
        year_month = lambda d: d['order_date'].dt.to_period('M'),
    )
)
print('Enriched shape:', enriched.shape)

# Step 3a: Monthly revenue + MoM
monthly = enriched.groupby('year_month')['revenue'].sum().reset_index()
monthly['mom_pct'] = monthly['revenue'].pct_change().mul(100).round(2)
print(monthly.tail(6).to_string(index=False))

# Step 3b: Top 5 products
top5 = enriched.groupby('product_name')['revenue'].sum().nlargest(5)
print('Top 5 products:'); print(top5.round(0))

# Step 3c: Channel x Region pivot
ch_reg = pd.pivot_table(enriched, values='revenue', index='region', columns='channel', aggfunc='sum', fill_value=0).round(0)
print(ch_reg)

# Step 3d: Cohort LTV
first_purchase = enriched.groupby('customer_id')['order_date'].min().dt.to_period('M').rename('cohort')
cohort_ltv = enriched.merge(first_purchase, on='customer_id').groupby('cohort')['revenue'].mean().round(2)
print('Avg LTV by cohort (first 6):'); print(cohort_ltv.head(6))

# Step 4: Executive summary
print('\n── EXECUTIVE SUMMARY ──')
print(f'Total Revenue:  ${enriched["revenue"].sum():,.0f}')
print(f'Total Orders:   {len(orders_df):,}')
print(f'Avg Order Value: ${enriched["revenue"].mean():.2f}')
print(f'Top Region:     {enriched.groupby("region")["revenue"].sum().idxmax()}')
print(f'Top Product:    {enriched.groupby("product_name")["revenue"].sum().idxmax()}')

out = os.path.join(tempfile.gettempdir(), 'enriched_sales.csv')
enriched.to_csv(out, index=False)
print(f'Exported to {out}')